In [ ]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import sympy as sp
import xarray as xr
from IPython.display import display,Math
warnings.filterwarnings('ignore')

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
KERNELFROM = CONFIGS['experiments']['sr']['runs']['sr_atm']['weightsfrom']
NNSEEDS    = CONFIGS['experiments']['nn']['seeds']
SRSEEDS    = CONFIGS['experiments']['sr']['seeds']
EQUATIONS  = CONFIGS['experiments']['sr']['optimizedeqs']
SPLIT      = 'test'
LANDFRAC   = 0.5
MINCOMPLEXITY = 8
RUNS       = {'sr_sfc':('sr_sfc_eq',[72,102]),'sr_all':('sr_all_eq',[102])}
REFERENCES = ['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']
NUMPYFUNCS = {'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,'abs':np.abs,'exp':np.exp,'max':np.maximum,'min':np.minimum}
SYMPYFUNCS = {'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,'abs':sp.Abs,'exp':sp.exp,'max':sp.Max,'min':sp.Min}
REGISTRY   = {row['name']:(row['form'],json.loads(row['constants'])) for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

In [ ]:
def kernel_integrate(fields,weights,dsig):
    return (fields*weights[None,:,:]*dsig[None,None,:]).sum(axis=2)

def prepare_form(form):
    return re.sub(r'\^','**',form)

def evaluate(form,columns,constants=None):
    namespace = dict(NUMPYFUNCS,__builtins__={})
    namespace.update(columns)
    namespace.update(constants or {})
    out = eval(prepare_form(form),namespace)
    return np.broadcast_to(np.asarray(out,dtype=float),columns['lf'].shape)

def to_precip(z):
    return np.expm1(STATS['tp_std']*np.maximum(z,0.0))

def calc_r2(pred,target,mask):
    return 1-np.sum((pred[mask]-target[mask])**2)/np.sum((target[mask]-target[mask].mean())**2)

def calc_skill(z):
    pred = to_precip(z)
    return {f'R2 {region}':calc_r2(pred,tp,mask) for region,mask in regionmasks.items()}

def to_physical(form,constants=None):
    expr = sp.sympify(prepare_form(form),locals={**SYMPYFUNCS,**PHYSICAL,**{name:sp.Float(value) for name,value in (constants or {}).items()}})
    return sp.N(STATS['tp_std']*expr,4)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    dsig    = ds['dsig'].values
    fields  = np.stack([ds[name].transpose('time','lat','lon','sig').values.reshape(-1,dsig.size) for name in FIELDVARS],axis=1)
    columns = {name:ds[name].transpose('time','lat','lon').values.ravel().astype(float) for name in ['shf','lhf']}
    columns['lf'] = xr.broadcast(ds['lf'],ds['shf'])[0].transpose('time','lat','lon').values.ravel().astype(float)
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    tp = ds['tp'].transpose('time','lat','lon').values.ravel().astype(float)

features = []
for seed in NNSEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'{KERNELFROM}_{seed}_weights.nc'),engine='h5netcdf') as ds:
        features.append(kernel_integrate(fields,ds['k'].values,dsig))
columns.update(dict(zip(FIELDVARS,np.mean(features,axis=0).T)))
columns['sr_atm_eq'] = evaluate(*REGISTRY['sr_atm_eq'][:1],columns,REGISTRY['sr_atm_eq'][1])
finite      = np.isfinite(tp)&np.all([np.isfinite(values) for values in columns.values()],axis=0)
regionmasks = {'Ocean':finite&(columns['lf']<LANDFRAC),'Land':finite&(columns['lf']>=LANDFRAC),'All':finite}

atmsymbol = sp.Symbol(r'\mathcal{A}')
PHYSICAL  = {'lf':sp.Symbol(r'\mathrm{LF}'),'sr_atm_eq':atmsymbol/STATS['tp_std']}
for name,label in [('rh',r'\widehat{\mathrm{RH}}'),('thetae',r'\widehat{\theta_e}'),('thetaestar',r'\widehat{\theta_e^*}'),('shf',r'\mathrm{SHF}'),('lhf',r'\mathrm{LHF}')]:
    PHYSICAL[name] = (sp.Symbol(label)-STATS[f'{name}_mean'])/STATS[f'{name}_std']

In [ ]:
rows = []
for name in REFERENCES:
    form,constants = REGISTRY[name]
    rows.append({'Source':'optimized','Run':EQUATIONS[name]['description'],'Seed':'--','Complexity':np.nan,'CSV Loss':np.nan,'Reference Loss':np.nan,
                 **calc_skill(evaluate(form,columns,constants)),'Equation':form,'Constants':constants})
for run,(eqname,refseeds) in RUNS.items():
    for seed in SRSEEDS:
        equations = pd.read_csv(os.path.join(MODELSDIR,'sr',f'{run}_{seed}_equations.csv'))
        refloss   = equations.loc[equations['complexity']==EQUATIONS[eqname]['refcomplexity'],'loss'].iloc[0] if seed in refseeds else np.nan
        for _,row in equations[equations['complexity']>=MINCOMPLEXITY].iterrows():
            rows.append({'Source':'pysr','Run':run,'Seed':seed,'Complexity':row['complexity'],'CSV Loss':row['loss'],'Reference Loss':refloss,
                         **calc_skill(evaluate(row['equation'],columns)),'Equation':row['equation'],'Constants':None})
candidates = pd.DataFrame(rows)
candidates['Loss <= Reference'] = np.where(candidates['Reference Loss'].isna(),'--',np.where(candidates['CSV Loss']<=candidates['Reference Loss'],'yes','no'))
display(candidates.drop(columns='Constants').sort_values('R2 All',ascending=False).style.hide(axis='index').format(precision=3))

In [ ]:
print('Physical exponent E, where P = (exp(E)-1)^+ and A = lambda*max(M,I)^3 is the SR-ATM exponent')
selected = candidates[(candidates['Source']=='optimized')|(candidates['Loss <= Reference']=='yes')|(candidates['R2 All']>=candidates.loc[candidates['Run']=='SR-ALL-PC','R2 All'].iloc[0])]
for _,row in selected.sort_values('R2 All',ascending=False).iterrows():
    print(f"{row['Run']} | seed {row['Seed']} | complexity {row['Complexity']} | R2 all {row['R2 All']:.3f}, ocean {row['R2 Ocean']:.3f}, land {row['R2 Land']:.3f}")
    display(Math(r'\mathcal{E} = '+sp.latex(to_physical(row['Equation'],row['Constants']))))